### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [9]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


model=init_chat_model("groq:qwen/qwen3.6-27b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000001842957FDD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000184295C4320>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="Movies rating out of  the 10")

In [12]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000001842957FDD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000184295C4320>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'Movies rating out of  the 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', '

In [14]:
response=model.invoke("Provide me the detail of the movie inception")
print(response.content)


<think>
Here's a thinking process:

1.  **Understand User Request**: The user wants "detail of the movie inception". This means they're asking for a comprehensive overview of the film "Inception" (2010), including plot, cast, director, themes, reception, etc.

2.  **Identify Key Information Needed**:
   - Title: Inception
   - Release Year: 2010
   - Director/Writer: Christopher Nolan
   - Genre: Sci-fi, Action, Heist, Psychological Thriller
   - Main Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Marion Cotillard, Michael Caine
   - Plot Summary: Core premise, main characters, mission structure, dream layers, climax, ending
   - Themes: Reality vs. dreams, guilt, subconscious, time dilation, memory, architecture of mind
   - Technical/Production Details: Cinematography, score, practical effects, filming locations
   - Reception: Box office, critical acclaim, awards/nominations
   - Legacy/Impact: Cultural influence, sequels, fan t

In [17]:
response=model_with_structure.invoke("Provide me the detail of the movie inception")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


### Message output along with parsed structure

In [19]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year movie was released")
    director:str=Field(..., description="Director of the movie")
    rating:float=Field(..., description="Movies rating out of  the 10")


model_with_structure=model.with_structured_output(Movie, include_raw=True)   

response=model_with_structure.invoke("Provide me the detail of the movie inception")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide me the detail of the movie inception"\n   - Key entity: Movie "Inception"\n   - Required information based on the available function: title, year, director, rating\n\n2.  **Identify Available Function:**\n   - Function: `Movie`\n   - Parameters: `title` (string, required), `year` (integer, required), `director` (string, required), `rating` (number, required)\n\n3.  **Gather Required Information for "Inception":**\n   - Title: Inception\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: I need to provide a rating out of 10. Common ratings for Inception: IMDb ~8.8, Rotten Tomatoes ~87%, Metacritic ~74. I\'ll use a widely recognized rating like 8.8 (IMDb scale out of 10) or just state a reasonable number. Since the function requires a number, I\'ll use 8.8.\n\n4.  **Check Function Requirements:**\n   - All required parame

### Nested Structure


In [36]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name:str
    role: str


class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None, description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide me the detail of the movie inception")   
response 

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles')], genres=['Sci-Fi', 'Action', 'Thriller', 'Adventure'], budget=160.0)

#### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.


In [ ]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str, ..., "The title of the movie"]
    year:Annotated[int, ..., "The year of the movie was release"]
    director:Annotated[str, ..., "The Director of the movie"]
    rating:Annotated[float, ..., "The movie rating of the 10"]


model_with_type_dict=model.with_structured_output(MovieDict)
response=model_with_type_dict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [50]:
class Actor(TypedDict):
    name:str
    role: str


class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None, description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide me the detail of the movie Avengers")   
response     

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [51]:
model_with_type_dict.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [54]:
model.profile

#### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the `@dataclass` decorator

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [61]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

llm = ChatGroq(model="qwen/qwen3.6-27b")
agent = create_agent(
    model=llm,  
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result




{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2c39daa9-33fb-46fb-95b6-383bbb40e22a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "John Doe, john@example.com, (555) 123-4567"\n   - Task: Extract contact info\n   - Available tool: `ContactInfo` with parameters `name`, `email`, `phone` (all required)\n\n2.  **Identify Parameters from Input:**\n   - Name: "John Doe"\n   - Email: "john@example.com"\n   - Phone: "(555) 123-4567"\n\n3.  **Validate Parameters against Tool Schema:**\n   - `name` (string, required): "John Doe" ✓\n   - `email` (string, required): "john@example.com" ✓\n   - `phone` (string, required): "(555) 123-4567" ✓\n\n4.  **Construct Tool Call:**\n   - Function: `ContactInfo`\n   - Arguments: `{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}`\

In [62]:
print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [63]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

llm = ChatGroq(model="qwen/qwen3.6-27b")

agent = create_agent(
    model=llm,  
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [67]:
## DataClass


from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

llm = ChatGroq(model="qwen/qwen3.6-27b")

agent = create_agent(
    model=llm,  
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)    

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')